In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from collections import Counter
import os

# Caminho para a Silver (saída do write_silver.py)
silver_path = "../../data/silver/cision_news_20251110.parquet"

if not os.path.exists(silver_path):
    raise FileNotFoundError(f" Ficheiro Silver não encontrado em {silver_path}")

df = pd.read_parquet(silver_path)
print(f"1Silver carregada com {len(df)} notícias únicas")
df.head(3)


In [ ]:
from sentence_transformers import SentenceTransformer
import time

# Criar amostra 10%
df_sample = df.sample(frac=0.10, random_state=42).reset_index(drop=True)

print(f"Amostra com {len(df_sample)} notícias")

# Carregar modelo
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Gerar embeddings
start = time.time()
embeddings = model.encode(
    df_sample["noticia_norm"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"Embeddings criados: {embeddings.shape}")
print(f"Tempo total: {(time.time()-start)/60:.1f} minutos")




In [ ]:
import umap

umap_reducer = umap.UMAP(n_components=10, random_state=42)
embeddings_reduced = umap_reducer.fit_transform(embeddings)

print(f"Dimensão original: {embeddings.shape[1]} → Reduzida: {embeddings_reduced.shape[1]}")


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Ks = [16, 18, 22]
scores = []

for k in Ks:
    labels = KMeans(n_clusters=k, random_state=42).fit_predict(embeddings_reduced)
    score = silhouette_score(embeddings_reduced, labels)
    print(f"k={k}: Silhouette={score:.3f}")
    scores.append((k, score))

best_k, best_score = max(scores, key=lambda x:x[1])
print(f"\nMelhor k = {best_k}, Silhouette = {best_score:.3f}")

df_sample["cluster_id"] = KMeans(n_clusters=best_k, random_state=42).fit_predict(embeddings_reduced)


In [ ]:
from sklearn.cluster import KMeans
from collections import Counter
import nltk
from nltk.corpus import stopwords

k_escolhido = 18 

# Treinar K-Means final
kmeans_final = KMeans(n_clusters=k_escolhido, random_state=42, n_init=10)
df_sample["cluster_id"] = kmeans_final.fit_predict(embeddings_reduced)

# Preparar stopwords PT
nltk.download('stopwords')
stop_words = set(stopwords.words("portuguese"))

print("\nPalavras mais comuns por cluster:")

for c in sorted(df_sample["cluster_id"].unique()):
    palavras = " ".join(df_sample[df_sample.cluster_id == c]["noticia_norm"])
    tokens = [p for p in palavras.split() if p not in stop_words and len(p) > 3]
    mais_comuns = Counter(tokens).most_common(15)

    print(f"\nCluster {c}:")
    print(", ".join([p[0] for p in mais_comuns]))





In [ ]:
# TF-IDF para extrair termos importantes por cluster

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

print("\n🔍 A calcular TF-IDF por cluster...\n")

# stopwords tem de ser LISTA para o TF-IDF
stop_words = list(stopwords.words("portuguese"))

vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words=stop_words,
    ngram_range=(1,2)   # inclui bigramas
)

for c in sorted(df_sample["cluster_id"].unique()):
    textos = df_sample[df_sample.cluster_id == c]["noticia_norm"].tolist()

    if len(textos) < 3:
        print(f"\nCluster {c}: (poucos textos)")
        continue

    # TF-IDF para o cluster
    tfidf_matrix = vectorizer.fit_transform(textos)
    scores = tfidf_matrix.mean(axis=0).A1
    termos = vectorizer.get_feature_names_out()

    # top termos
    top_idx = scores.argsort()[::-1][:15]
    top_termos = [termos[i] for i in top_idx]

    print(f"\n Cluster {c} — termos TF-IDF mais relevantes:")
    print(", ".join(top_termos))


In [ ]:
print("------ CLUSTER 0 ------")
print("\n".join(df_sample[df_sample.cluster_id == 0]["noticia_norm"].head(10)))

print("\n------ CLUSTER 3 ------")
print("\n".join(df_sample[df_sample.cluster_id == 3]["noticia_norm"].head(10)))



In [ ]:
print("\n------ CLUSTER 1 ------")
print("\n".join(df_sample[df_sample.cluster_id == 1]["noticia_norm"].head(10)))

print("\n------ CLUSTER 2 ------")
print("\n".join(df_sample[df_sample.cluster_id == 2]["noticia_norm"].head(10)))


In [ ]:
print("\n------ CLUSTER 4 ------")
print("\n".join(df_sample[df_sample.cluster_id == 4]["noticia_norm"].head(10)))

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Carregar variáveis do ficheiro .env
load_dotenv()

# Ler a chave a partir do .env
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

dicionario_cluster_para_nome = {}

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Diz apenas 'Ligação OK'"}]
)

print(response.choices[0].message.content)



In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
import pandas as pd # Necessário para o .join na extração

# --- PARTE 1: EXTRAÇÃO DE PALAVRAS-CHAVE (TF/IDF SIMPLIFICADO) ---

# Garantir stopwords em português
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')
stop_words = set(stopwords.words('portuguese'))


cluster_palavras = {}

for c in sorted(df_sample["cluster_id"].unique()):
    # Junta todo o texto de um cluster
    palavras = " ".join(df_sample[df_sample.cluster_id == c]["noticia_norm"].astype(str))

    # Tokens filtrados (remove stopwords e pontuação)
    tokens = [
        p.lower()
        for p in palavras.split()
        if p.lower() not in stop_words and len(p) > 2
    ]

    # Guarda as 15 palavras mais comuns
    mais_comuns = [p[0] for p in Counter(tokens).most_common(15)]
    cluster_palavras[c] = mais_comuns

print("✔ Palavras mais frequentes extraídas por cluster.\n")


# --- PARTE 2: ROTULAGEM DOS CLUSTERS (USANDO GPT-4o-mini E 19 FEW-SHOTS) ---

# Lista RÍGIDA das 19 categorias de política pública (A sua taxonomia)
areas_oficiais = """
Agricultura e Floresta; Administração Pública; Ambiente e Clima; Cultura; Defesa;
Desporto; Economia; Educação e Formação; Energia; Habitação; Impostos e taxas;
I&D e Inovação; Justiça; Mar e Pescas; Proteção Social; Saúde; Segurança;
Trabalho; Transportes e Mobilidade
"""

dicionario_cluster_para_nome = {}
# ⚠️ Assume-se que 'client' (OpenAI) está definido e ativo.
print("A rotular os clusters (GPT-4o-mini)...")

# Construção dos 19 exemplos Few-Shot (o prompt extenso)
few_shot_examples = """
Exemplos de Classificação (Obrigatório):

Palavras: hospitais, médicos, SNS, urgências, cuidados de saúde, enfermeiros, vacinação
Categoria: Saúde

Palavras: futebol, campeonato, liga, equipa, golo, estádio, atleta, modalidade, jogos
Categoria: Desporto

Palavras: agricultura, floresta, gado, vinha, produção agrícola, incêndios rurais, campo
Categoria: Agricultura e Floresta

Palavras: tribunais, juiz, advogados, julgamento, processo, procurador, lei, justiça
Categoria: Justiça

Palavras: inflação, investimento, PIB, mercado, exportação, bancos, impostos, finanças, economia
Categoria: Economia

Palavras: câmara municipal, autarquia, serviços públicos, concurso, funcionários, estado
Categoria: Administração Pública

Palavras: clima, aquecimento global, poluição, sustentabilidade, emissões, reciclagem
Categoria: Ambiente e Clima

Palavras: museu, teatro, cinema, artes, património, espetáculo, festival, cultura
Categoria: Cultura

Palavras: defesa nacional, forças armadas, exército, marinha, aeronáutica, missão militar
Categoria: Defesa

Palavras: escola, ensino, alunos, professores, universidade, formação profissional, aulas
Categoria: Educação e Formação

Palavras: energia, petróleo, gás, eletricidade, renovável, solar, eólica, transmissão
Categoria: Energia

Palavras: habitação, renda, imobiliário, arrendamento, casa, obras, construção, crédito
Categoria: Habitação

Palavras: impostos, IVA, IRS, taxas, contribuições, autoridade tributária, fiscal
Categoria: Impostos e taxas

Palavras: investigação, inovação, ciência, startup, tecnologia, laboratório, descoberta
Categoria: I&D e Inovação

Palavras: mar, pescas, oceano, porto, pescadores, costa, navio, recursos marinhos
Categoria: Mar e Pescas

Palavras: segurança social, pensões, reformas, apoios sociais, subsídio, idosos
Categoria: Proteção Social

Palavras: segurança, PSP, GNR, polícia, crime, violência, roubo, investigação, assalto
Categoria: Segurança

Palavras: trabalho, emprego, salário, desemprego, contrato, empresa, horário, sindicato
Categoria: Trabalho

Palavras: transportes, mobilidade, carro, comboio, avião, autocarro, metro, estrada, aeroporto
Categoria: Transportes e Mobilidade
"""

for c, palavras in cluster_palavras.items():

    prompt = f"""
Classifica o seguinte grupo de palavras-chave numa categoria de políticas públicas portuguesas.

Regras de Classificação:
- A resposta TEM de ser UMA das categorias listadas em "Categorias disponíveis".
- Escolhe apenas UMA categoria, a mais representativa.
- Não cries categorias novas.
- Responde APENAS com o nome da categoria.

{few_shot_examples}

Categorias disponíveis:
{areas_oficiais}

Palavras do cluster:
{', '.join(palavras)}

Categoria:
"""
    # ... Resto da chamada API para o GPT (client.chat.completions.create)
    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    categoria_nome = resposta.choices[0].message.content.strip()
    dicionario_cluster_para_nome[c] = categoria_nome

    print(f"Cluster {c} ➜ {categoria_nome}")

print("\n✔ Nomes de categorias gerados com sucesso!")


   

In [ ]:
import pandas as pd

df_gold = pd.read_csv("../../data_sample/golden_set_10_por_categoria.csv")


In [ ]:
# confirmar distribuição
df_gold["Categoria"].value_counts()


In [ ]:
df_gold[["Categoria", "id", "titulo"]]
